# 🧬 Notebook 03: Genome Alignment (Mapping)

## 🎯 Introduction
Now that our RNA-Seq reads are trimmed and of high quality, the next crucial step is to figure out where these reads came from. To do this, we will align (map) them to the *Mycobacterium tuberculosis* reference genome.

### Objectives for this Notebook:
1. Download the official reference genome (`.fasta` or `.fna`) and its annotation file (`.gff` or `.gtf`).
2. Install the alignment tool **Bowtie2**.
3. Build a genome index for Bowtie2.
4. Align our cleaned single-end reads to the reference genome to generate `.sam` files.

In [ ]:
%%bash
# 1. Create a dedicated directory for the reference genome
mkdir -p reference
cd reference

# 2. Download the fasta file (Genome Sequence)
echo "📥 Downloading Reference Genome (FASTA)..."
wget -qO M_tuberculosis_H37Rv.fna.gz https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/195/955/GCF_000195955.2_ASM19595v2/GCF_000195955.2_ASM19595v2_genomic.fna.gz

# 3. Download the GFF/GTF file (Genome Annotation)
echo "📥 Downloading Annotation (GFF)..."
wget -qO M_tuberculosis_H37Rv.gff.gz https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/195/955/GCF_000195955.2_ASM19595v2/GCF_000195955.2_ASM19595v2_genomic.gff.gz

echo "✅ Download Complete! Files saved in 'reference/' directory."

In [ ]:
%%bash
# 1. Unzip the reference genome FASTA file
echo "🔓 Unzipping the reference genome..."
gunzip -f reference/M_tuberculosis_H37Rv.fna.gz

# 2. Create a directory for the Bowtie2 index files
mkdir -p reference/bowtie2_index

# 3. Build the genome index
echo "🏗️ Building Bowtie2 index (this might take a minute)..."
bowtie2-build \
  reference/M_tuberculosis_H37Rv.fna \
  reference/bowtie2_index/Tuberculosis_index

echo "🎉 Genome Indexing is complete! Ready for mapping."

### 🎯 Step 2: Genome Alignment (Mapping) using Bowtie2 & Samtools
In this step, we map our high-quality trimmed single-end reads to the *Mycobacterium tuberculosis* H37Rv reference genome to identify their genomic origins.

**Our Pipeline Workflow:**
1. **Alignment (`bowtie2`):** Aligns the trimmed `.fastq.gz` reads against the pre-built genome index using the `-U` flag (for unpaired/single-end data).
2. **Compression & Sorting (`samtools view | samtools sort`):** Converts the massive text-based `.sam` file into a highly compressed, coordinate-sorted binary `.bam` file.
3. **BAM Indexing (`samtools index`):** Generates a `.bai` index file for rapid data retrieval during downstream analysis.
4. **Storage Optimization (`rm`):** Automatically deletes the temporary intermediate `.sam` files to save disk space on the virtual machine.

In [ ]:
%%bash
# 1. Create directory for alignment results
mkdir -p results/mapped_data

# 2. Run Bowtie2 and Samtools loop for single-end reads
for FILE_PATH in results/trimmed_data/*_trimmed.fastq.gz; do
    
    # Extract sample name
    BASE_NAME=$(basename "$FILE_PATH" _trimmed.fastq.gz)
    
    echo "🎯 Aligning sample: ${BASE_NAME}..."
    
    # Run bowtie2 and pipe the output directly to samtools to save space
    bowtie2 \
      -x reference/bowtie2_index/Tuberculosis_index \
      -U results/trimmed_data/${BASE_NAME}_trimmed.fastq.gz \
      --threads 4 \
      -S results/mapped_data/${BASE_NAME}.sam
      
    echo "📦 Converting SAM to Sorted BAM for: ${BASE_NAME}..."
    # Convert SAM to BAM, sort it, and save it
    samtools view -bS results/mapped_data/${BASE_NAME}.sam | samtools sort -o results/mapped_data/${BASE_NAME}_sorted.bam
    
    # Index the final BAM file (creates a .bai file for fast browsing)
    samtools index results/mapped_data/${BASE_NAME}_sorted.bam
    
    # Clean up the huge SAM file to save disk space
    rm results/mapped_data/${BASE_NAME}.sam

done
echo "🎉 Mapping and sorting completed successfully for all samples!"

## 📌 Conclusion
The Genome Alignment phase using Bowtie2 was highly successful:
* **Alignment Rate:** All samples achieved an exceptional alignment rate of over **99%** against the *Mycobacterium tuberculosis* H37Rv genome.
* **Mapping Quality:** Approximately **98%** of the reads mapped uniquely (`aligned exactly 1 time`), which is ideal for accurate downstream quantification.
* **Data Verification:** Bowtie2 logs confirmed that **100%** of the reads were unpaired, validating our single-end pipeline configuration.

The coordinate-sorted binary files (`.bam`) and their indexes (`.bai`) are now safely stored in `results/mapped_data/`, ready for gene expression quantification.